# Sketch to Shoes Images Using Conditional Latend Diffusion Models (CLDMs)

In [ ]:
!pip install pytorch-fid
!pip install pytorch-msssim



### Imports


In [ ]:
import zipfile, glob, os, json, torch, torch.nn as nn, torch.nn.functional as F, torchvision.transforms as T, itertools, math, shutil
import matplotlib.pyplot as plt, numpy as np, random

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from typing import Callable, Optional, Tuple, List
from diffusers import AutoencoderKL
from tqdm.auto import tqdm
from google.colab import drive
from torchvision.utils import save_image

from pytorch_fid import fid_score
from pytorch_msssim import ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
drive.mount('/content/drive')

### Kaggle API Credentials
Set dummy Kaggle credentials to download dataset

In [ ]:
# Export to environment so Kaggle CLI/Python API can use them directly
os.environ['KAGGLE_USERNAME'] = "dummy_name"
os.environ['KAGGLE_KEY'] = "dummy_key"

# Also write kaggle.json for compatibility
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
creds_path = kaggle_dir / 'kaggle.json'
with open(creds_path, 'w') as f:
    json.dump({'username': "dummy_name", 'key': "dummy_key"}, f)
os.chmod(creds_path, 0o600)

### Download Dataset
Download SketchToImage dataset off Kaggle.

In [ ]:
# Local dataset folder
DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = 'balraj98/edges2shoes-dataset'
ZIP_PATH = DATA_ROOT / 'edges2shoes.zip'
EXTRACT_DIR = DATA_ROOT / 'edges2shoes'
TRAIN_DIR = EXTRACT_DIR / 'train'
VAL_DIR   = EXTRACT_DIR / 'val'

# Download and extract only if not already extracted
if not EXTRACT_DIR.exists():
    print("Downloading dataset…")
    !kaggle datasets download -d {DATASET_SLUG} -p {DATA_ROOT} -o

    # Find downloaded zip
    zips = sorted(DATA_ROOT.glob('*.zip'))
    if not zips:
        raise FileNotFoundError('No zip files downloaded from Kaggle')
    zip_file = zips[0]
    print('Found zip:', zip_file)

    # Extract locally
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extraction complete.")

    # Randomly move X images into val to correct split (need more val items for FID)
    train_images = list(TRAIN_DIR.glob("*.jpg"))
    to_copy = train_images[:6000]
    for img in to_copy:
        shutil.move(img, VAL_DIR / img.name)

    print(f"Copied {len(to_copy)} images from train to val")
    print("New train count:", len(list(TRAIN_DIR.glob("*.jpg"))))
    print("Val count:", len(list(VAL_DIR.glob("*.jpg"))))
else:
    print("Dataset already extracted. Skipping download.")

print('Train images dir:', TRAIN_DIR)
print('Train image count:', len(list(TRAIN_DIR.glob('*.jpg'))))

### Preprocess Dataset
Preprocesses Kaggle Dataset. This includes downsampling of the sketch from 256 by 256 to 32 by 32, as well as moving the real images to their latent space (avoid repeating for training).

In [ ]:
# --- Paths ---
PREPROCESS_LOCAL = Path("/content/data/preprocessed")
PREPROCESS_DRIVE = Path("/content/drive/MyDrive/edges2shoes_preprocessed")
TRAIN_OUT = PREPROCESS_LOCAL / "train"
VAL_OUT   = PREPROCESS_LOCAL / "val"
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
VAL_OUT.mkdir(parents=True, exist_ok=True)
train_images = list(TRAIN_DIR.glob("*.jpg"))
val_images   = list(VAL_DIR.glob("*.jpg"))

# --- Pretrained VAE from Stability AI ---
pretrained_vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
pretrained_vae.eval()
print("Loaded Pre-Trained VAE")

def decode_latent(vae, lat, scale_factor, is_custom):
    # Unscale the latent tensor before passing it to the VAE decoder
    unscaled_lat = lat / scale_factor
    output = vae.decode(unscaled_lat)

    if is_custom:
        # Custom VAE (assumed to return a raw Tensor)
        return output.clamp(0, 1)
    else:
        # Pre-trained VAE (returns DecoderOutput object with .sample)
        return output.sample.clamp(0, 1)

def process_image(img_path, vae, scale_factor, show_images=False, custom=False):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    sketch = img.crop((0, 0, w//2, h))
    real = img.crop((w//2, 0, w, h))

    sketch_gray = sketch.convert("L")
    sketch_t = T.ToTensor()(sketch_gray).unsqueeze(0)
    real_t = T.ToTensor()(real).unsqueeze(0).to(device)

    # 5. Encode real image to latent
    with torch.no_grad():
        z = None
        if custom:
          # Custom VAE (mu/logvar)
          mu, logvar = vae.encode(real_t)
          z = vae.reparameterize(mu, logvar)
        else:
          # Pre-trained SD VAE (latent_dist.sample)
          z = vae.encode(real_t).latent_dist.sample()

        # Apply scaling factor *after* encoding
        z = z * scale_factor

    # --- Display ---
    if show_images:
      # Use the robust helper function for decoding
      with torch.no_grad():
          real_decoded = decode_latent(vae, z, scale_factor, custom)

      fig, axes = plt.subplots(1, 4, figsize=(15, 4))

      axes[0].imshow(sketch_gray, cmap="gray"); axes[0].set_title("Original Sketch (1ch)"); axes[0].axis("off")
      axes[1].imshow(real); axes[1].set_title("Real Image"); axes[1].axis("off")

      real_img = real_decoded.squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1)
      axes[2].imshow(real_img); axes[2].set_title("Decoded Latent"); axes[2].axis("off")

      latent_img = z.squeeze(0).mean(0).cpu()
      axes[3].imshow(latent_img, cmap="viridis"); axes[3].set_title("Latent Tensor (Scaled)"); axes[3].axis("off")
      plt.tight_layout()
      plt.show()

    # Returns scaled latent
    return sketch_t.squeeze(0), z.squeeze(0).cpu()

# --- Execution Logic ---
PREPROCESS_ZIP = PREPROCESS_DRIVE / "edges2shoes_preprocessed.zip"

# Check If Preprocessed Files Exist On Drive
if PREPROCESS_ZIP.exists(): # If They Do, Copy From Drive
    print("Preprocessed zip found on Drive. Copying and extracting locally...")
    shutil.copy(PREPROCESS_ZIP, PREPROCESS_LOCAL / PREPROCESS_ZIP.name)
    with zipfile.ZipFile(PREPROCESS_LOCAL / PREPROCESS_ZIP.name, 'r') as zip_ref:
        zip_ref.extractall(PREPROCESS_LOCAL)
    print("Extraction complete.")
else: # If They Don't, Preprocess
    print("No preprocessed zip found. Running preprocessing...")

    def preprocess_split(images, out_dir, show_first_n=2):
      count = 0
      for img_path in images:
          try:
              show = count < show_first_n
              sketch_latent, real_latent = process_image(img_path, pretrained_vae, 0.18215, show_images=show)
              torch.save({"sketch": sketch_latent, "real": real_latent}, out_dir / f"{img_path.stem}.pt")

              count += 1
              if count % 2000 == 0:
                  print(f"Processed {count} images in {out_dir.name}...")

          except Exception as e:
              print(f"Skipping {img_path}: {e}")

    # Preprocess both train and test splits with Stable Diffusion VAE
    print("Preprocessing train split...")
    preprocess_split(train_images, TRAIN_OUT)

    print("Preprocessing val split...")
    preprocess_split(val_images, VAL_OUT)

    # --- Zip and Copy to Drive ---
    print("Preprocessing done. Zipping files...")
    zip_path = PREPROCESS_LOCAL / "edges2shoes_preprocessed.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in PREPROCESS_LOCAL.rglob("*.pt"):
            zipf.write(file_path, arcname=file_path.name)

    if not os.path.exists(PREPROCESS_DRIVE):
        os.makedirs(PREPROCESS_DRIVE, exist_ok=True)
        print(f"Created Drive directory: {PREPROCESS_DRIVE}")

    shutil.copy(zip_path, PREPROCESS_DRIVE / zip_path.name)
    print(f"Preprocessed zip saved to Drive: {PREPROCESS_DRIVE / zip_path.name}")

print("Total Processed In Train:", len(list((TRAIN_OUT).glob("*.pt"))))
print("Total Processed In Val:", len(list((VAL_OUT).glob("*.pt"))))

### Data Loader
Implements a PyTorch `Dataset` that splits each 512x256 paired image into `(sketch, real)` halves of size 256x256.

In [ ]:
class Edges2ShoesPairsPreprocessed(Dataset):
    def __init__(self, root_dir: str | os.PathLike):
        self.root_dir = Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"{self.root_dir} does not exist")
        self.files = sorted(self.root_dir.rglob("*.pt"))
        if not self.files:
            raise FileNotFoundError(f"No preprocessed files found in {self.root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        data = torch.load(self.files[idx])
        return {
            'sketch': data['sketch'],  # sketch (1,256,256)
            'image': data['real'],     # latent real image (4,32,32)
            'path': str(self.files[idx])
        }

# --- Instantiate dataset and dataloader ---
BATCH_SIZE = 64
NUM_WORKERS = 0 # increasing this will flood the console with exceptions during training

train_dataset = Edges2ShoesPairsPreprocessed(TRAIN_OUT)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Number of samples: {len(train_dataset)}, batch size: {BATCH_SIZE}")

In [ ]:
if __name__ == '__main__':
  # Quick sanity check: iterate one batch and show shapes
  batch = next(iter(train_loader))
  print('Batch keys:', list(batch.keys()))
  print('Sketch tensor:', batch['sketch'].shape, batch['sketch'].dtype, batch['sketch'].min().item(), batch['sketch'].max().item())
  print('Image tensor:', batch['image'].shape, batch['image'].dtype, batch['image'].min().item(), batch['image'].max().item())

### Diffusion Setup

In [ ]:
# --- Diffusion setup in latent space ---
timesteps = 800  # Total timesteps
beta_start = 1e-4   # the smallest about of noise added at the first step t = 1
beta_end   = 0.02   # the largest amount of noise that can be added at t = timesteps = 200

betas = torch.linspace(beta_start, beta_end, timesteps).to(device) # creates the schedules of noise from min to max at timesteps evenly spaced intervals
alphas = 1.0 - betas                                       # calculate the alpha schedules by minusing every beta from 1
alphas_cumprod = torch.cumprod(alphas, dim=0)              # calculates the cumulative product of all alpha values up to timestep t
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]], dim=0) # calculates cumulative values at step c - 1

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)       # needed for inference, but not training

def q_sample(z0, t, noise=None): # takes in sample and level of noise t
    if noise is None:
        noise = torch.randn_like(z0) # generate noise tensor the same size as z0, by sampling from a normal distribution
    sqrt_ac = sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) # get scaling factor for input image
    sqrt_om = sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) # get scaling factor for noise
    return sqrt_ac * z0 + sqrt_om * noise # return noised tensor image

def sample_timesteps(batch_size): # returns a tensor of batch size (64) numbers between 0 and timesteps
    return torch.randint(low=0, high=timesteps, size=(batch_size,), device=device)

def sinusoidal_time_embedding(timesteps_tensor, dim): # takes in a vector of batch size (64) timesteps and returns their embeddings
    device_ = timesteps_tensor.device
    half_dim = dim // 2
    emb = math.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=device_) * -emb)
    emb = timesteps_tensor.float().unsqueeze(1) * emb.unsqueeze(0)
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb


### Conditional U-Net Class For Latent Space

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Residual Block For U-Net ---
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        # FiLM conditioning for time
        self.cond_proj = nn.Linear(cond_dim, 2 * out_ch)

        # Identity or projection for skip connection
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        # Time conditioning
        gamma, beta = self.cond_proj(cond).chunk(2, dim=1)
        gamma = gamma[..., None, None]
        beta = beta[..., None, None]

        h = self.conv1(F.silu(self.norm1(x)))
        # Apply FiLM before the final activation/conv of the block
        h = self.conv2(F.silu(self.norm2(h) * (1 + gamma) + beta))

        return h + self.skip(x)

# --- Sketch Encoder (Learns Downsampling) ---
class SketchEncoder(nn.Module):
    """Processes 256x256 sketch into multi-scale feature maps."""
    def __init__(self, base_ch=32):
        super().__init__()
        # 256 -> 128 -> 64 -> 32
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base_ch, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1), nn.SiLU(),
            nn.Conv2d(base_ch, base_ch, 3, stride=2, padding=1), nn.SiLU()
        )
        # 32 -> 16
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_ch, base_ch * 2, 3, stride=2, padding=1), nn.SiLU()
        )
        # 16 -> 8
        self.enc3 = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch * 4, 3, stride=2, padding=1), nn.SiLU()
        )

    def forward(self, x):
        feat32 = self.enc1(x)    # [B, 32, 32, 32]
        feat16 = self.enc2(feat32) # [B, 64, 16, 16]
        feat8  = self.enc3(feat16) # [B, 128, 8, 8]
        return feat32, feat16, feat8

# --- Latent U-Net ---
class LatentUNet(nn.Module):
    def __init__(self, in_ch=4, base_ch=64, time_dim=64):
        super().__init__()
        self.time_dim = time_dim

        # 1. Sketch Encoder
        self.sketch_enc = SketchEncoder(base_ch=32)
        sk_ch = 32 # Channel depth from sketch encoder outputs

        # 2. Conditioning MLP (Time)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # --- ENCODER PATH ---
        # Input: Latent(4) + Sketch_Feat(32) = 36
        self.conv_in = nn.Conv2d(in_ch + sk_ch, base_ch, 3, padding=1) # [36, 32, 32] -> [64, 32, 32]
        self.down1 = ResidualBlock(base_ch, base_ch, time_dim) # [64, 32, 32] -> [64, 32, 32]

        # Strided Conv instead of Pool: base_ch(64) + Sketch_Feat(64) = 128
        self.down2_conv = nn.Conv2d(base_ch, base_ch * 2, 3, stride=2, padding=1) # [64, 32, 32] -> [128, 16, 16]
        self.down2 = ResidualBlock(base_ch * 2 + (sk_ch * 2), base_ch * 2, time_dim) # [192, 16, 16] -> [128, 16, 16]

        # Strided Conv instead of Pool: base_ch*2(128) + Sketch_Feat(128) = 256
        self.down3_conv = nn.Conv2d(base_ch * 2, base_ch * 2, 3, stride=2, padding=1) # [128, 16, 16] -> [128, 8, 8]
        self.down3 = ResidualBlock(base_ch * 2 + (sk_ch * 4), base_ch * 2, time_dim) # [256, 8, 8] -> [128, 8, 8]

        # Bottleneck
        self.mid = ResidualBlock(base_ch * 2, base_ch * 2, time_dim) # [128, 8, 8] -> [128, 8, 8]

        # --- DECODER PATH (using PixelShuffle for upsampling) ---
        self.up_conv3 = nn.Sequential(nn.Conv2d(base_ch * 2, base_ch * 4, 1), nn.PixelShuffle(2)) # [128, 8, 8] -> [256, 8, 8] -> [64, 16, 16]
        self.up3 = ResidualBlock(base_ch * 1 + base_ch * 2, base_ch * 2, time_dim) # [192, 16, 16] -> [128, 16, 16]

        self.up_conv2 = nn.Sequential(nn.Conv2d(base_ch * 2, base_ch * 4, 1), nn.PixelShuffle(2)) # [128, 16, 16] -> [256, 16, 16] -> [64, 32, 32]
        self.up2 = ResidualBlock(base_ch * 2, base_ch, time_dim) # [128, 32, 32] -> [64, 32, 32]

        self.up1 = ResidualBlock(base_ch + base_ch, base_ch, time_dim) # [128, 32, 32] -> [64, 32, 32]

        # Output
        self.conv_out = nn.Conv2d(base_ch, in_ch, 3, padding=1) # [64, 32, 32] -> [4, 32, 32]

    def forward(self, x, t, sketch_256):
        # 1. Timestep conditioning
        ste = sinusoidal_time_embedding(t, self.time_dim)
        cond = self.time_mlp(ste)

        # 2. Sketch multi-scale features
        sk32, sk16, sk8 = self.sketch_enc(sketch_256)

        # --- Downsampling ---
        x0 = self.conv_in(torch.cat([x, sk32], dim=1)) # concat sketch embedding to noised latent : [36, 32, 32] -> [64, 32, 32]
        x1 = self.down1(x0, cond)              # Add : [64, 32, 32] -> [64, 32, 32]

        x2_in = self.down2_conv(x1)            # [64, 32, 32] -> [128, 16, 16]
        x2 = self.down2(torch.cat([x2_in, sk16], dim=1), cond) # [192, 16, 16] -> [128, 16, 16] (works as we concat sk16 - 64 channels)

        x3_in = self.down3_conv(x2)            # [128, 16, 16] -> [128, 8, 8]
        x3 = self.down3(torch.cat([x3_in, sk8], dim=1), cond)  # [256, 8, 8] -> [128, 8, 8]

        m = self.mid(x3, cond)                 # Bottleneck [128, 8, 8] -> [128, 8, 8]

        # --- Upsampling ---
        u3 = self.up_conv3(m)                  # [128, 8, 8] -> [64, 16, 16]
        u3 = self.up3(torch.cat([u3, x2], dim=1), cond) # [192, 16, 16] -> [128, 16, 16] (works as we concat x2 which has 128 channels)

        u2 = self.up_conv2(u3)                 # [128, 16, 16] -> [64, 32, 32]
        u2 = self.up2(torch.cat([u2, x1], dim=1), cond) # [192, 32, 32] -> [64, 32, 32] (FIX THE INPUT TO THIS LAYER IS ACTUALLY 128x32x32 ())

        u1 = self.up1(torch.cat([u2, x0], dim=1), cond) # [128, 32, 32] -> [64, 32, 32] (works as x0 has 64 channels)

        return self.conv_out(u1)

# --- Initialization ---
latent_channels = 4
base_channels = 64
time_dimension = 64

unet = LatentUNet(in_ch=latent_channels, base_ch=base_channels, time_dim=time_dimension).to(device)
opt = torch.optim.AdamW(unet.parameters(), lr=2e-4)

print("Parameters (M):", sum(p.numel() for p in unet.parameters()) / 1e6)

### Train U-Net with Pre-Trained VAE

In [ ]:
import os
import warnings
import torch.nn.functional as F
import torch

warnings.filterwarnings("ignore", ".*does not have many workers.*") # suppress warnings so that we do not flood concole as num workers > 0

# --- Hyperparameters and Paths (Assumed to be defined globally) ---
RUN_ID = 7
DRIVE_PATH = f'/content/drive/MyDrive/diffusion_checkpoints/run_{RUN_ID}'
os.makedirs(DRIVE_PATH, exist_ok=True)
num_diff_epochs = 30
unet.train()

# --- Sketch Dropout Configuration ---
# Helps the model learn to denoise even when the condition is noisy or absent.
SKETCH_DROPOUT_PROB = 0.1

for epoch in range(num_diff_epochs):
    epoch_loss = 0.0
    n = 0

    pbar = tqdm(train_loader, desc=f"Diff Epoch {epoch+1}/{num_diff_epochs}")
    for batch in pbar:
        # z0 is the clean latent image [B, 4, 32, 32]
        z0 = batch['image'].to(device)

        # sketches is the FULL-RESOLUTION sketch [B, 1, 256, 256] (Crucial change for new U-Net)
        sketches = batch['sketch'].to(device)

        # --- Sketch Dropout Implementation ---
        # Generate a mask where elements are True with SKETCH_DROPOUT_PROB
        mask = torch.rand(sketches.size(0), 1, 1, 1, device=device) < SKETCH_DROPOUT_PROB

        # Replace the sketches with a tensor of zeros (blank image) where the mask is True
        # This forces the UNet to sometimes ignore the sketch.
        sketches = sketches.masked_fill(mask, 0.0)

        t = sample_timesteps(z0.size(0)).to(device)
        noise = torch.randn_like(z0)
        z_t = q_sample(z0, t, noise)

        # The UNet now expects the FULL-RESOLUTION sketch
        noise_pred = unet(z_t, t.float(), sketches)

        loss = F.mse_loss(noise_pred, noise)

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss += loss.item()
        n += 1

        pbar.set_postfix({"loss": loss.item()})

    avg = epoch_loss / n
    print(f"Epoch {epoch+1}: avg batch loss = {avg:.4f}")

    # --- SAVE CHECKPOINT EVERY 20 EPOCHS ---
    if (epoch + 1) % 20 == 0:
        filename = f'latent_unet_checkpoint_epoch_{epoch+1}.pth'
        save_path = os.path.join(DRIVE_PATH, filename)

        checkpoint = {
            'unet_state_dict': unet.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'epoch': epoch + 1,
            'loss': loss.item()
        }

        torch.save(checkpoint, save_path)
        print(f"   💾 Saved Checkpoint: {filename}")

# --- FINAL SAVE ---
FINAL_PATH = os.path.join(DRIVE_PATH, 'latent_unet_checkpoint_final.pth')
checkpoint = {
    'unet_state_dict': unet.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'epoch': num_diff_epochs,
    'loss': loss.item()
}
torch.save(checkpoint, FINAL_PATH)

print(f"\n✅ Final model saved to: {FINAL_PATH}")

### Test Inference
Apply inference on test sample to generate real image.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, sketch_latent, T_steps, latent_h=32, latent_w=32):
    """
    Performs the full denoising process from pure noise z_T down to z_0.

    Args:
        model (nn.Module): The trained LatentUNet model (epsilon predictor).
        sketch_latent (torch.Tensor): The 1-channel downsampled sketch image (B, 1, 32, 32).
        T_steps (int): Total number of timesteps (e.g., 1000).
        latent_h (int): Latent image height.
        latent_w (int): Latent image width.
    """
    device = sketch_latent.device
    batch_size = sketch_latent.size(0)

    # Start with pure noise
    z = torch.randn(batch_size, 4, latent_h, latent_w, device=device)

    for t in tqdm(reversed(range(1, T_steps + 1)), desc="Denoising", total=T_steps):
        t_index = t - 1

        # 1. Create time tensor
        t_tensor = torch.full((batch_size,), t_index, device=device, dtype=torch.float32)

        predicted_noise = model(z, t_tensor, sketch_latent) # Updated function call

        sqrt_recip_alpha_t = sqrt_recip_alphas[t_index].view(-1, 1, 1, 1)
        beta_t = betas[t_index].view(-1, 1, 1, 1)
        sqrt_om_t = sqrt_one_minus_alphas_cumprod[t_index].view(-1, 1, 1, 1)

        # Calculate mean of the posterior distribution (mu_theta)
        mu_theta = sqrt_recip_alpha_t * (z - beta_t * predicted_noise / sqrt_om_t)

        if t > 1:
            # Sample from the posterior distribution
            variance = posterior_variance[t_index].view(-1, 1, 1, 1)
            noise = torch.randn_like(z)
            z = mu_theta + torch.sqrt(variance) * noise
        else:
            z = mu_theta  # Final step is deterministic (z_0)

    return z # z_0 (the predicted latent image)


def load_checkpoint(filepath, unet, opt=None):
    """Loads the state dictionaries for the UNet and Optimizer from the checkpoint file."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Checkpoint not found at: {filepath}")

    print(f"Loading checkpoint from {filepath}...")
    checkpoint = torch.load(filepath, map_location=device)

    unet.load_state_dict(checkpoint['unet_state_dict'])

    if opt is not None and 'optimizer_state_dict' in checkpoint:
        opt.load_state_dict(checkpoint['optimizer_state_dict'])

    unet.eval() # Set model to evaluation mode
    print("UNet loaded successfully. Ready for inference.")

    return unet


In [ ]:
# --- Model and Path Definitions ---
RUN_ID = "7"
DRIVE_PATH = f'/content/drive/MyDrive/diffusion_checkpoints/run_{RUN_ID}/latent_unet_checkpoint_final.pth'
VAL_PREPROCESSED_DIR = "/content/data/preprocessed/val"
VAL_ORIGINAL_DIR    = "/content/data/edges2shoes/val"
OUTPUT_DIR = '/content/diffusion_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_val_item(pt_path):
    data = torch.load(pt_path)

    sketch_down = data["sketch"]          # [1, 256, 256]
    actual_latent = data["real"]          # [4, 32, 32]

    img_name = Path(pt_path).stem + ".jpg"
    img = Image.open(os.path.join(VAL_ORIGINAL_DIR, img_name)).convert("RGB")
    w, h = img.size

    sketch_orig = img.crop((0, 0, w//2, h)).convert("L")
    real_orig   = img.crop((w//2, 0, w, h))

    return sketch_down, actual_latent, sketch_orig, real_orig

def run_test_inference(unet_model, vae_model, scale_factor, val_preprocessed_dir, custom, diffusion_output, show_images=5):
    pt_files = sorted(Path(val_preprocessed_dir).glob("*.pt"))[:100]
    if not pt_files:
        print("Error: No preprocessed val files found.")
        return

    for idx, pt_path in enumerate(pt_files, 1):
        if idx % 200 == 0:
            print(f"Processed {idx} files.")

        # Assume load_val_item is defined and works
        sketch_down, actual_latent, sketch_orig, real_orig = load_val_item(pt_path)

        sketch_down = sketch_down.unsqueeze(0).to(device)
        actual_latent = actual_latent.unsqueeze(0).to(device)

        with torch.no_grad():
            # --- Generate Predicted Latent ---
            predicted_latent = p_sample_loop(
                unet_model, sketch_down, T_steps=timesteps
            )

            # --- Decode Images using robust helper ---
            predicted_image = decode_latent(vae_model, predicted_latent, scale_factor, custom)
            actual_decoded_image = decode_latent(vae_model, actual_latent, scale_factor, custom)

            out_file = Path(diffusion_output) / f"{pt_path.stem}_decoded.pt"
            torch.save({"real": real_orig, "generated": actual_decoded_image}, out_file)

            if idx <= show_images:
              # ... (plotting code remains similar) ...
              fig, axes = plt.subplots(1, 6, figsize=(16, 8))

              # NOTE: Plotting assumes sketch_orig/real_orig are [C, H, W] or [1, C, H, W]
              axes[0].imshow(sketch_orig, cmap="gray"); axes[0].axis("off")
              axes[1].imshow(real_orig); axes[1].axis("off")
              axes[2].imshow(actual_latent.squeeze(0).mean(0).cpu(), cmap="viridis"); axes[2].axis("off")
              axes[3].imshow(actual_decoded_image.squeeze(0).permute(1,2,0).cpu()); axes[3].axis("off")
              axes[4].imshow(predicted_latent.squeeze(0).mean(0).cpu(), cmap="viridis"); axes[4].axis("off")
              axes[5].imshow(predicted_image.squeeze(0).permute(1,2,0).cpu()); axes[5].axis("off")

              plt.tight_layout()
              plt.show()

# Load checkpoint
load_checkpoint(DRIVE_PATH, unet, opt if 'opt' in globals() else None)

# Run Inference
run_test_inference(unet, pretrained_vae, 0.18215, VAL_PREPROCESSED_DIR, False, OUTPUT_DIR)


# Custom VAE


### VAE Encoder and Decoder

In [ ]:
# Adding Residuals in the EncBlock and UpBlock

class EncBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)

    def forward(self, x):
        # First conv downsamples and changes channels
        h = self.down(x)                     # (B, out_ch, H/2, W/2)
        residual = h                         # skip after channel projection
        h = F.silu(self.norm1(h))
        h = self.conv2(h)
        h = F.silu(self.norm2(h))
        return h + residual                  # ResNet-style skip


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)

    def forward(self, x):
        # x is already upsampled
        h = self.conv1(x)
        h = F.silu(self.norm1(h))
        h = self.conv2(h)
        h = F.silu(self.norm2(h))
        # project for residual if needed
        if h.shape[1] != x.shape[1]:
            # simple 1x1 conv to match channels
            res = F.conv2d(x, weight=torch.zeros(h.shape[1], x.shape[1], 1, 1, device=x.device))
        else:
            res = x
        return h + res


class ConvVAE(nn.Module):
    """
    3x256x256 -> 4x32x32 -> 3x256x256
    Encoder: 3 -> 64 -> 128 -> 256 (down to 32x32)
    Decoder: 4 -> 256 -> 128 -> 64 -> 3
    """
    def __init__(self, img_channels=3, latent_channels=4, base_ch=64):
        super().__init__()
        self.img_channels = img_channels
        self.latent_channels = latent_channels

        # ---------- Encoder ----------
        # 256 -> 128 -> 64 -> 32
        self.enc1 = EncBlock(img_channels, base_ch)        # 3 -> 64, 256->128
        self.enc2 = EncBlock(base_ch, base_ch * 2)         # 64 -> 128, 128->64
        self.enc3 = EncBlock(base_ch * 2, base_ch * 4)     # 128 -> 256, 64->32

        # keep 32x32, produce 2 * latent_channels (mu + logvar)
        self.enc_conv_mu_logvar = nn.Conv2d(base_ch * 4, latent_channels * 2,
                                            kernel_size=3, stride=1, padding=1)

        # ---------- Decoder ----------
        # latent (4,32,32) -> 256 channels at 32x32
        self.dec_conv_in = nn.Conv2d(latent_channels, base_ch * 4, 3, padding=1)
        self.dec_norm_in = nn.GroupNorm(8, base_ch * 4)

        # Up blocks: each preceded by interpolate x2
        self.dec_up1 = UpBlock(base_ch * 4, base_ch * 2)   # 32 -> 64
        self.dec_up2 = UpBlock(base_ch * 2, base_ch)       # 64 -> 128
        self.dec_up3 = UpBlock(base_ch, base_ch)           # 128 -> 256

        # Final conv to image channels
        self.dec_conv_out = nn.Conv2d(base_ch, img_channels, 3, padding=1)

    # ---- Encoder: return mu, logvar with shape (B, 4, 32, 32) ----
    def encode(self, x):
        # x: (B, 3, 256, 256)
        h = self.enc1(x)                       # (B,64,128,128)
        h = self.enc2(h)                       # (B,128,64,64)
        h = self.enc3(h)                       # (B,256,32,32)
        h = self.enc_conv_mu_logvar(h)         # (B,8,32,32)
        mu, logvar = torch.chunk(h, 2, dim=1)  # (B,4,32,32) each
        return mu, logvar

    # ---- Reparameterization trick ----
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std  # (B, 4, 32, 32)

    # ---- Decoder: z (B, 4, 32, 32) -> x_hat (B, 3, 256, 256) ----
    def decode(self, z):
        # 32x32
        h = self.dec_conv_in(z)
        h = F.silu(self.dec_norm_in(h))        # (B,256,32,32)

        # 32 -> 64
        h = F.interpolate(h, scale_factor=2, mode="nearest")  # (B,256,64,64)
        h = self.dec_up1(h)                                   # (B,128,64,64)

        # 64 -> 128
        h = F.interpolate(h, scale_factor=2, mode="nearest")  # (B,128,128,128)
        h = self.dec_up2(h)                                   # (B,64,128,128)

        # 128 -> 256
        h = F.interpolate(h, scale_factor=2, mode="nearest")  # (B,64,256,256)
        h = self.dec_up3(h)                                   # (B,64,256,256)

        h = self.dec_conv_out(h)                              # (B,3,256,256)
        x_hat = torch.sigmoid(h)                              # [0,1]
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


### Loss Function and Optimizer

In [ ]:
def conv_vae_loss(x, x_hat, mu, logvar, lambda_rec=1.0, lambda_ssim=0.5, beta_kl=0.5):
    # x, x_hat in [0,1], shape (B,3,256,256)

    # Reconstruction: L1
    rec_l1 = F.l1_loss(x_hat, x, reduction="mean")

    # SSIM: we want 1 - SSIM (so lower is better)
    ssim_val = ssim(x_hat, x, data_range=1.0, size_average=True)
    rec_ssim = 1.0 - ssim_val

    # KL divergence
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total = lambda_rec * rec_l1 + lambda_ssim * rec_ssim + beta_kl * kl
    return total, rec_l1, rec_ssim, kl

custom_vae = ConvVAE(img_channels=3, latent_channels=4, base_ch=64).to(device)
custom_vae_opt = torch.optim.Adam(custom_vae.parameters(), lr=1e-4)

print("Custom VAE params (M):", sum(p.numel() for p in custom_vae.parameters()) / 1e6)


### Data Loader For Custom VAE

In [ ]:
from torchvision import transforms as T

vae_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),          # [0,1]
])

class ShoesRealImages(Dataset): # data loader to load shoe image
    def __init__(self, root_dir):
        self.paths = sorted(Path(root_dir).glob("*.jpg"))
        self.transform = vae_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        w, h = img.size
        real = img.crop((w//2, 0, w, h))  # right half: real shoe
        real_t = self.transform(real)     # (3,256,256)
        return real_t

real_dataset = ShoesRealImages(TRAIN_DIR)
real_loader = DataLoader(real_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

if __name__ == '__main__':
  # Quick sanity check: iterate one batch and show shapes
  batch = next(iter(real_loader))
  print("Batch shape:", batch.shape)  # [batch_size, 3, 256, 256]
  print("min/max:", batch.min().item(), batch.max().item())

### Custom VAE Training Loop

In [ ]:
import os, warnings
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", ".*does not have many workers.*")

RUN_ID_VAE = 6
VAE_DRIVE_PATH = f"/content/drive/MyDrive/vae_checkpoints/run_{RUN_ID_VAE}"
os.makedirs(VAE_DRIVE_PATH, exist_ok=True)

resume_epoch = 0                # change this value to load in previous weights and continue training from there
resume_checkpoint = None
num_vae_epochs = 51
epoch_checkpoint = 5

# Look for most recent checkpoint using full Drive paths
checkpoints = [f for f in os.listdir(VAE_DRIVE_PATH) if f.startswith('conv_vae_checkpoint_epoch_')]
checkpoints_full = [os.path.join(VAE_DRIVE_PATH, f) for f in checkpoints]

if checkpoints_full:
    latest_ckpt = max(checkpoints_full, key=lambda x: int(x.split('_epoch_')[1].split('.pth')[0]))
    print(f"Found latest checkpoint: {os.path.basename(latest_ckpt)}")

    try:
        resume_checkpoint = torch.load(latest_ckpt, map_location=device)
        resume_epoch = resume_checkpoint['epoch']
        print(f"Resuming from epoch {resume_epoch}")
    except Exception as e:
        print(f"Error loading checkpoint {latest_ckpt}: {e}")
        print("Starting from scratch instead.")
else:
    print("No previous checkpoints found. Starting from scratch.")

# Load if resuming
if resume_checkpoint:
    custom_vae.load_state_dict(resume_checkpoint['vae_state_dict'])
    custom_vae_opt.load_state_dict(resume_checkpoint['optimizer_state_dict'])
    print("✅ Loaded VAE and optimizer states")
else:
    print("Starting fresh training")

# Initialize running sums for the new loss terms
running_loss = running_recon_l1 = running_recon_ssim = running_kl = 0.0
n_batches = 0

# ---- Training Loop ----
custom_vae.train()
start_epoch = resume_epoch
for epoch in range(start_epoch, num_vae_epochs):
    running_loss = running_recon_l1 = running_recon_ssim = running_kl = 0.0
    n_batches = 0

    pbar = tqdm(real_loader, desc=f"VAE Epoch {epoch+1}/{num_vae_epochs}")
    for real_imgs in pbar:
        real_imgs = real_imgs.to(device)

        custom_vae_opt.zero_grad()
        x_hat, mu, logvar = custom_vae(real_imgs)
        loss, recon_l1, recon_ssim, kl = conv_vae_loss(real_imgs, x_hat, mu, logvar)
        loss.backward()
        custom_vae_opt.step()

        # Update running sums with new variable names
        running_loss += loss.item()
        running_recon_l1 += recon_l1.item()
        running_recon_ssim += recon_ssim.item()
        running_kl += kl.item()
        n_batches += 1

        pbar.set_postfix({"loss": loss.item(), "l1": recon_l1.item()})

    # Calculate averages
    avg_loss = running_loss / n_batches
    avg_recon_l1 = running_recon_l1 / n_batches
    avg_recon_ssim = running_recon_ssim / n_batches
    avg_kl = running_kl / n_batches

    print(f"[VAE] Epoch {epoch+1:03d} | Loss {avg_loss:.4f} | L1 {avg_recon_l1:.4f} | SSIM {avg_recon_ssim:.4f} | KL {avg_kl:.4f}")

    # SAVE CHECKPOINT EVERY x EPOCHS
    if (epoch + 1) % epoch_checkpoint == 0:
        filename = f"conv_vae_checkpoint_epoch_{epoch+1}.pth"
        save_path = os.path.join(VAE_DRIVE_PATH, filename)
        checkpoint = {
            "vae_state_dict": custom_vae.state_dict(),
            "optimizer_state_dict": custom_vae_opt.state_dict(),
            "epoch": epoch + 1,
            "loss": avg_loss,
        }
        torch.save(checkpoint, save_path)
        print(f"   Saved VAE checkpoint: {filename}")

# FINAL SAVE (unchanged)
FINAL_VAE_PATH = os.path.join(VAE_DRIVE_PATH, "conv_vae_checkpoint_final.pth")
checkpoint = {
    "vae_state_dict": custom_vae.state_dict(),
    "optimizer_state_dict": custom_vae_opt.state_dict(),
    "epoch": num_vae_epochs,
    "loss": avg_loss,
}
torch.save(checkpoint, FINAL_VAE_PATH)
print(f"\n Final VAE saved to: {FINAL_VAE_PATH}")


### Visualise Reconstructed Images Of Custom VAE

In [ ]:
custom_vae.eval()
real_batch = next(iter(real_loader)).to(device)

with torch.no_grad():
    x_hat, mu, logvar = custom_vae(real_batch)

real_batch = real_batch.cpu()
x_hat = x_hat.cpu()

n = 6
plt.figure(figsize=(2*n, 4))
for i in range(n):
    # original
    plt.subplot(2, n, i+1)
    plt.imshow(real_batch[i].permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title("Orig")

    # reconstruction
    plt.subplot(2, n, n+i+1)
    plt.imshow(x_hat[i].permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title("Recon")

plt.tight_layout()
plt.show()

# Train U-Net On Custom VAE

### Calculate Scale Factor Of Custom VAE

In [ ]:
def calculate_vae_latent_std(vae, dataloader, device, num_batches=20):
    vae.eval()
    all_latents = []
    print("Calculating latent statistics...")
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_batches: break
            # batch is [32, 3, 256, 256] from real_loader
            imgs = batch.to(device)
            mu, logvar = vae.encode(imgs)
            z = vae.reparameterize(mu, logvar)
            all_latents.append(z.cpu())

    combined_latents = torch.cat(all_latents, dim=0)
    latent_std = combined_latents.std().item()
    print(f"Latent Standard Deviation: {latent_std:.4f}")
    print(f"Recommended Scale Factor (1/std): {1.0 / latent_std:.4f}")
    return 1.0 / latent_std


CUSTOM_VAE_SCALE = calculate_vae_latent_std(custom_vae, real_loader, device)

### Preprocess Data With Custom VAE

In [ ]:
# --- Paths for CUSTOM VAE preprocessing ---
PREPROCESS_LOCAL_CUSTOM = Path("/content/data/preprocessed_custom")
PREPROCESS_DRIVE_CUSTOM = Path("/content/drive/MyDrive/edges2shoes_preprocessed_custom")
PREPROCESS_ZIP_CUSTOM = PREPROCESS_DRIVE_CUSTOM / "edges2shoes_preprocessed.zip"
TRAIN_OUT_CUSTOM = PREPROCESS_LOCAL_CUSTOM / "train"
VAL_OUT_CUSTOM   = PREPROCESS_LOCAL_CUSTOM / "val"
TRAIN_OUT_CUSTOM.mkdir(parents=True, exist_ok=True)
VAL_OUT_CUSTOM.mkdir(parents=True, exist_ok=True)
train_images = list(TRAIN_DIR.glob("*.jpg"))
val_images   = list(VAL_DIR.glob("*.jpg"))

# --- Trained Custom ConvVAE ---
custom_vae.eval()
print("Using custom ConvVAE for preprocessing")

if PREPROCESS_ZIP_CUSTOM.exists():
    print("Custom preprocessed zip found on Drive. Copying and extracting locally...")
    shutil.copy(PREPROCESS_ZIP_CUSTOM, PREPROCESS_LOCAL_CUSTOM / PREPROCESS_ZIP_CUSTOM.name)
    with zipfile.ZipFile(PREPROCESS_LOCAL_CUSTOM / PREPROCESS_ZIP_CUSTOM.name, 'r') as zip_ref:
        zip_ref.extractall(PREPROCESS_LOCAL_CUSTOM)
    print("Extraction complete.")
else:
    print("No custom preprocessed zip found. Running custom preprocessing...")

    os.makedirs(PREPROCESS_LOCAL_CUSTOM, exist_ok=True)

    def preprocess_split(images, out_dir, show_first_n=2):
      count = 0
      for img_path in images:
          try:
              show = count < show_first_n
              sketch_latent, real_latent = process_image(img_path, custom_vae, CUSTOM_VAE_SCALE, show_images=show, custom=True)
              torch.save({"sketch": sketch_latent, "real": real_latent}, out_dir / f"{img_path.stem}.pt")

              count += 1
              if count % 2000 == 0:
                  print(f"Processed {count} images in {out_dir.name}...")

          except Exception as e:
              print(f"Skipping {img_path}: {e}")

    # Preprocess both train and test splits with Stable Diffusion VAE
    print("Preprocessing train split with Custom VAE...")
    preprocess_split(train_images, TRAIN_OUT_CUSTOM)

    print("Preprocessing val split with Custom VAE...")
    preprocess_split(val_images, VAL_OUT_CUSTOM)

    # --- Zip and Copy to Drive ---
    print("Preprocessing done. Zipping files...")
    zip_path = PREPROCESS_LOCAL_CUSTOM / "edges2shoes_preprocessed.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in PREPROCESS_LOCAL_CUSTOM.rglob("*.pt"):
            zipf.write(file_path, arcname=file_path.name)

    if not os.path.exists(PREPROCESS_DRIVE_CUSTOM):
        os.makedirs(PREPROCESS_DRIVE_CUSTOM, exist_ok=True)
        print(f"Created Drive directory: {PREPROCESS_DRIVE_CUSTOM}")

    shutil.copy(zip_path, PREPROCESS_DRIVE_CUSTOM / zip_path.name)
    print(f"Preprocessed zip saved to Drive: {PREPROCESS_DRIVE_CUSTOM / zip_path.name}")

print("Total Processed In Train:", len(list((TRAIN_OUT_CUSTOM).glob("*.pt"))))
print("Total Processed In Val:", len(list((VAL_OUT_CUSTOM).glob("*.pt"))))

### Data Loader For Dataset Preprocessed With Custom VAE

In [ ]:
class Edges2ShoesPairsPreprocessed(Dataset):
    def __init__(self, root_dir: str | os.PathLike):
        self.root_dir = Path(root_dir)
        if not self.root_dir.exists():
            raise FileNotFoundError(f"{self.root_dir} does not exist")
        self.files = sorted(self.root_dir.glob("*.pt"))
        if not self.files:
            raise FileNotFoundError(f"No preprocessed files found in {self.root_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        data = torch.load(self.files[idx])
        return {
            'sketch': data['sketch'],  # sketch (1,256,256)
            'image': data['real'],     # latent real image (4,32,32)
            'path': str(self.files[idx])
        }

# --- Instantiate dataset and dataloader ---
BATCH_SIZE = 64
NUM_WORKERS = 0 # increasing this will flood the console with exceptions during training

custom_vae_train_dataset = Edges2ShoesPairsPreprocessed(TRAIN_OUT_CUSTOM)
custom_vae_train_loader = DataLoader(
    custom_vae_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

batch = next(iter(custom_vae_train_loader))
print('Batch keys:', list(batch.keys()))
print('Sketch tensor:', batch['sketch'].shape, batch['sketch'].dtype, batch['sketch'].min().item(), batch['sketch'].max().item())
print('Image tensor:', batch['image'].shape, batch['image'].dtype, batch['image'].min().item(), batch['image'].max().item())
print(f"Number of samples: {len(custom_vae_train_dataset)}, batch size: {BATCH_SIZE}")


### Train U-Net In Custom VAE Latent Space

In [ ]:
# --- Hyperparameters and Paths (Assumed to be defined globally) ---
DRIVE_PATH = f'/content/drive/MyDrive/diffusion_checkpoints_custom/run_{RUN_ID}'
os.makedirs(DRIVE_PATH, exist_ok=True)
num_diff_epochs = 30

# U-net Shape
latent_channels = 4
base_channels = 64
time_dimension = 64

custom_vae_unet = LatentUNet(in_ch=latent_channels, base_ch=base_channels, time_dim=time_dimension).to(device)
custom_vae_unet.train()
custom_opt = torch.optim.AdamW(custom_vae_unet.parameters(), lr=2e-4)

print("Parameters (M):", sum(p.numel() for p in unet.parameters()) / 1e6)

# --- Sketch Dropout Configuration ---
SKETCH_DROPOUT_PROB = 0.1     # Helps the model learn to denoise even when the condition is noisy or absent.

for epoch in range(num_diff_epochs):
    epoch_loss = 0.0
    n = 0

    pbar = tqdm(custom_vae_train_loader, desc=f"Diff Epoch {epoch+1}/{num_diff_epochs}")
    for batch in pbar:
        # z0 is the clean latent image [B, 4, 32, 32]
        z0 = batch['image'].to(device)

        # sketches is the FULL-RESOLUTION sketch [B, 1, 256, 256] (Crucial change for new U-Net)
        sketches = batch['sketch'].to(device)

        # --- Sketch Dropout Implementation ---
        # Generate a mask where elements are True with SKETCH_DROPOUT_PROB
        mask = torch.rand(sketches.size(0), 1, 1, 1, device=device) < SKETCH_DROPOUT_PROB

        # Replace the sketches with a tensor of zeros (blank image) where the mask is True
        # This forces the UNet to sometimes ignore the sketch.
        sketches = sketches.masked_fill(mask, 0.0)

        t = sample_timesteps(z0.size(0)).to(device)
        noise = torch.randn_like(z0)
        z_t = q_sample(z0, t, noise)

        # The UNet now expects the FULL-RESOLUTION sketch
        noise_pred = custom_vae_unet(z_t, t.float(), sketches)

        loss = F.mse_loss(noise_pred, noise)

        custom_opt.zero_grad()
        loss.backward()
        custom_opt.step()

        epoch_loss += loss.item()
        n += 1

        pbar.set_postfix({"loss": loss.item()})

    avg = epoch_loss / n
    print(f"Epoch {epoch+1}: avg batch loss = {avg:.4f}")

    # --- SAVE CHECKPOINT EVERY 20 EPOCHS ---
    if (epoch + 1) % 20 == 0:
        filename = f'latent_unet_checkpoint_epoch_{epoch+1}.pth'
        save_path = os.path.join(DRIVE_PATH, filename)

        checkpoint = {
            'unet_state_dict': unet.state_dict(),
            'optimizer_state_dict': custom_opt.state_dict(),
            'epoch': epoch + 1,
            'loss': loss.item()
        }

        torch.save(checkpoint, save_path)
        print(f"   💾 Saved Checkpoint: {filename}")

# --- FINAL SAVE ---
FINAL_PATH = os.path.join(DRIVE_PATH, 'latent_unet_checkpoint_final.pth')
checkpoint = {
    'unet_state_dict': unet.state_dict(),
    'optimizer_state_dict': custom_opt.state_dict(),
    'epoch': num_diff_epochs,
    'loss': loss.item()
}
torch.save(checkpoint, FINAL_PATH)

print(f"\n✅ Final model saved to: {FINAL_PATH}")

### Inference On Custom VAE Trained U-NET

In [ ]:
# --- Model and Path Definitions ---
DRIVE_PATH = f'/content/drive/MyDrive/diffusion_checkpoints_custom/run_{RUN_ID}/latent_unet_checkpoint_final.pth'
VAL_PREPROCESSED_CUSTOM_DIR = "/content/data/preprocessed_custom/val"
VAL_ORIGINAL_DIR    = "/content/data/edges2shoes/val"
OUTPUT_DIR = '/content/diffusion_output_custom'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load checkpoint
load_checkpoint(DRIVE_PATH, unet, opt if 'opt' in globals() else None)

# Run Inference
run_test_inference(custom_vae_unet, custom_vae, CUSTOM_VAE_SCALE, VAL_PREPROCESSED_CUSTOM_DIR, True, OUTPUT_DIR)


# Test FID Of Both U-Nets

In [ ]:
import torch
from pytorch_fid.fid_score import calculate_fid_given_paths
from torchvision.utils import save_image
from pathlib import Path
import os
import shutil

def calculate_fid_metrics(unet_model, vae_model, scale_factor, is_custom, val_preprocessed_dir, fid_output_dir):

    print(f"Starting FID calculation for model (Custom={is_custom})")

    # 1. Setup Directories
    GEN_DIR = Path(fid_output_dir) / ('custom_gen' if is_custom else 'stable_gen')
    REAL_DIR = Path(fid_output_dir) / 'real' # Real images are the same for both models

    if GEN_DIR.exists():
        shutil.rmtree(GEN_DIR) # Clear previous generated images
    GEN_DIR.mkdir(parents=True, exist_ok=True)

    REAL_DIR.mkdir(parents=True, exist_ok=True)

    # 2. Run Inference and Save Images
    pt_files = sorted(Path(val_preprocessed_dir).glob("*.pt"))[:100]
    if not pt_files:
        print("Error: No preprocessed val files found.")
        return

    unet_model.eval()
    vae_model.eval()

    print(f"Processing {len(pt_files)} files for image generation...")

    with torch.no_grad():
        for idx, pt_path in enumerate(pt_files):
            if idx % 500 == 0:
                print(f"Processed {idx}/{len(pt_files)} files for FID.")

            # Load data from the preprocessed file
            sketch_down, actual_latent, sketch_orig, real_orig = load_val_item(pt_path)

            # Add batch dim and move to device
            sketch_down = sketch_down.unsqueeze(0).to(device)

            # --- Generate Predicted Latent ---
            predicted_latent = p_sample_loop(
                unet_model, sketch_down, T_steps=timesteps
            )

            # --- Decode Predicted Image ---
            # NOTE: decode_latent must be defined/available in the scope where this function is used
            predicted_image = decode_latent(vae_model, predicted_latent.to(device), scale_factor, is_custom)

            # Save Real Image (only need to do this once per run)
            real_img_path = REAL_DIR / f"{pt_path.stem}.png"
            from torchvision.transforms.functional import to_tensor

            # Save Real Image (only need to do this once per run)
            real_img_path = REAL_DIR / f"{pt_path.stem}.png"
            if not real_img_path.exists():
                if not torch.is_tensor(real_orig):
                    real_orig = to_tensor(real_orig)
                save_image(real_orig.unsqueeze(0).cpu(), real_img_path)


            # Save Generated Image
            gen_img_path = GEN_DIR / f"{pt_path.stem}.png"
            # predicted_image is [1, 3, 256, 256]


            save_image(predicted_image.cpu(), gen_img_path)

    # 3. Calculate FID
    print("Generation complete. Calculating FID...")
    paths = [str(REAL_DIR), str(GEN_DIR)]
    fid_value = calculate_fid_given_paths(paths, batch_size=64, device=device, dims=2048)

    print(f"\n--- Result ---")
    print(f"FID Score for model (Custom={is_custom}): {fid_value:.4f}")

    return fid_value


In [ ]:
# Pretrained VAE
fid = calculate_fid_metrics(
    unet_model=unet,
    vae_model=pretrained_vae,
    scale_factor=0.18215,
    is_custom=False,
    val_preprocessed_dir="/content/data/preprocessed/val",
    fid_output_dir="fid_outputs"
)

# Custom VAE
fid = calculate_fid_metrics(
    unet_model=custom_vae_unet,
    vae_model=custom_vae,
    scale_factor=CUSTOM_VAE_SCALE,
    is_custom=True,
    val_preprocessed_dir="/content/data/preprocessed_custom/val",
    fid_output_dir="fid_outputs_custom"
)
